# Agentic Platform: Agent Evaluation

How do you know if your agent is *good*? This lab builds two automated evaluation approaches from scratch:

| Approach | What it measures | How it works |
|----------|-----------------|--------------|
| **Assertion-Based** | Response correctness | An LLM judge checks each assertion (TRUE/FALSE) against the agent's output |
| **Step-Based** | Path efficiency | Compares the agent's actual tool-call sequence against expected steps using "logical groupings" |

**Why both?** An agent can get the right answer via a terrible path (10 redundant searches), or take an elegant path but hallucinate the final answer. You need both lenses.

**What you'll build in this notebook:**
1. A researcher agent with web search (Tavily)
2. An `AssertionEvaluator` that uses an LLM judge to score responses
3. An `AgentEvalHarness` that measures step efficiency via logical groupings
4. Summary utilities to track metrics over time

# Architecture Overview

**TL;DR:** Two evaluation approaches — *Assertion-Based* checks if the response meets qualitative criteria (TRUE/FALSE per assertion via an LLM judge), while *Step-Based* checks if the agent took a reasonable path to get there (comparing actual tool calls against expected steps). Both are automated, parallelizable, and CI/CD-ready.

![Agent Evaluation Types](../images/evals-type.png)

## Setup: Build a Research Agent to Evaluate

Before we can evaluate an agent, we need one. Below we create a simple researcher agent using PydanticAI + Tavily web search. This agent:
- Takes a natural-language question
- Searches the web via Tavily
- Returns a structured `ResearchResponse` (content + source URLs)

This is the "subject under test" — the agent we'll evaluate in Parts 1 and 2.

In [ ]:
from pydantic_ai import Agent
from pydantic import BaseModel
from tavily import TavilyClient
from typing import Dict
import os
from dotenv import load_dotenv

from agentic_platform.core.models.memory_models import ToolResult
from mcp import ListToolsResult
from mcp.server import FastMCP
from typing import List, Any

load_dotenv()

# First we wrap our own MCP server in a MCPServerStdio object
from pydantic_ai.mcp import MCPServerStdio as PyAIMCPServerStdio

# Load our API key from the environment variable
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')

# Now lets create our research tools
class WebSearch(BaseModel):
    query: str

server = FastMCP()

def search_web(query: WebSearch) -> List[Dict[str, Any]]:
    '''Search the web to get back a list of results and content.'''
    client: TavilyClient = TavilyClient(os.getenv("TAVILY_KEY"))
    response: Dict[str, any] = client.search(query=query.query)
    return response['results']



In [ ]:
from labs_common import HAIKU_STRANDS_ID
# Next lets create our researcher agent. 
from pydantic_ai import Agent as PyAIAgent
from labs_common import LabsBasePrompt as BasePrompt
from pydantic import BaseModel

RESEARCHER_SYSTEM_PROMPT = """
You are a helpful research agent with web search capabilities. Your job is to:

1. Search for accurate, up-to-date information about any topic
2. Provide clear and conscise answers to the users question. Provide a source annotation at the end of your response that maps to the source links below.
3. Cite your sources with numbered links at the end of your response.

Always be factual and objective in your research. Be clear and concise in your responses.
Only answer the immediate question, you do not need to provide any additional context or commentary.
If someone asks who the CEO of a company is, you should just say their name for example.
"""

# Define the expected output structure
class ResearchResponse(BaseModel):
    content: str
    sources: list[str]

# Create the agent with the simplified prompt
researcher_agent = PyAIAgent(
    HAIKU_STRANDS_ID,
    system_prompt=RESEARCHER_SYSTEM_PROMPT,
    result_type=ResearchResponse,
)

researcher_agent.tool_plain(search_web)

In [ ]:
import nest_asyncio
nest_asyncio.apply()

test_case = "Who is the CEO of Amazon?"

result = researcher_agent.run_sync(test_case)

result.output



In [ ]:
# Print out the conversation history to see where the results came from. 
result.all_messages()

# Part 1: Assertion-Based Evaluation

**Concept:** Define a set of assertions (natural-language claims) about what the agent's response *should* contain. A separate LLM ("the judge") reads the agent's output and scores each assertion as TRUE or FALSE.

**How the code below works:**
1. `TestCase` — holds a query + list of assertion strings
2. `AssertionEvaluator._check_assertion()` — sends each assertion to Bedrock (Haiku) with the agent's response, gets back TRUE/FALSE
3. `evaluate_test_cases()` — runs the agent, then scores all assertions, returns a `TestResult` per query

Frameworks like Ragas and Pydantic Evals do this for you, but seeing it from scratch makes the mechanics clear.

In [ ]:
from labs_common import HAIKU_MODEL_ID
import asyncio
import json
import boto3
from typing import List, Callable, Awaitable
from pydantic import BaseModel

class TestCase(BaseModel):
    name: str
    query: str
    assertions: List[str]

class AgentResponse(BaseModel):
    content: str
    sources: List[str]
    success: bool
    error: str = None

class AssertionResult(BaseModel):
    assertion: str
    passed: bool

class TestResult(BaseModel):
    name: str
    query: str
    assertions_results: List[AssertionResult]
    agent_failed: bool = False
    agent_error: str = None
    
    @property
    def total_assertions(self) -> int:
        return len(self.assertions_results)
    
    @property
    def passed_assertions(self) -> int:
        return sum(1 for r in self.assertions_results if r.passed)
    
    @property
    def success_rate(self) -> float:
        if self.agent_failed or self.total_assertions == 0:
            return 0.0
        return self.passed_assertions / self.total_assertions

class AssertionEvaluator:
    def __init__(self, model_id: str = HAIKU_MODEL_ID, region: str = 'us-east-1'):
        self.bedrock = boto3.client('bedrock-runtime', region_name=region)
        self.model_id = model_id
    
    def _check_assertion(self, query: str, response: AgentResponse, assertion: str) -> bool:
        prompt = f"""Check if this assertion is TRUE or FALSE based on the research response.
        
        QUERY: {query}
        RESPONSE: {response.content}
        SOURCES: {response.sources}
        
        ASSERTION TO CHECK: {assertion}
        
        Return only "TRUE" or "FALSE" (no explanation needed)."""
        
        try:
            resp = self.bedrock.converse(
                modelId=self.model_id,
                messages=[{"role": "user", "content": [{"text": prompt}]}],
                inferenceConfig={"maxTokens": 50, "temperature": 0}
            )
            content = resp['output']['message']['content'][0]['text'].strip().upper()
            return "TRUE" in content
        except Exception as e:
            print(f"Error checking assertion: {e}")
            return False
    
    async def evaluate_test_case(self, test_case: TestCase, agent_function: Callable[[str], Awaitable[AgentResponse]]) -> TestResult:
        print(f"\n{'='*70}")
        print(f"🧪 TEST: {test_case.name}")
        print(f"{'='*70}")
        print(f"\n📝 Query:\n   {test_case.query}")
        
        response = await agent_function(test_case.query)
        
        if not response.success:
            print(f"\n❌ Agent failed: {response.error}")
            return TestResult(
                name=test_case.name,
                query=test_case.query,
                assertions_results=[],
                agent_failed=True,
                agent_error=response.error
            )
        
        print(f"\n💬 Response:\n   {response.content}")
        print(f"\n🔗 Sources:")
        for src in response.sources:
            print(f"   • {src}")
        
        print(f"\n{'─'*70}")
        print(f"   Checking {len(test_case.assertions)} assertions...")
        print(f"{'─'*70}")
        
        assertion_results = []
        for i, assertion in enumerate(test_case.assertions, 1):
            passed = self._check_assertion(test_case.query, response, assertion)
            symbol = "✅" if passed else "❌"
            print(f"   {symbol} [{i}] {assertion}")
            assertion_results.append(AssertionResult(assertion=assertion, passed=passed))
        
        passed_count = sum(1 for r in assertion_results if r.passed)
        print(f"\n   Result: {passed_count}/{len(assertion_results)} passed")
        
        return TestResult(
            name=test_case.name,
            query=test_case.query,
            assertions_results=assertion_results
        )
    
    async def evaluate_test_cases(self, test_cases: List[TestCase], agent_function: Callable[[str], Awaitable[AgentResponse]]) -> List[TestResult]:
        results = []
        for test_case in test_cases:
            result = await self.evaluate_test_case(test_case, agent_function)
            results.append(result)
        return results
    
    @staticmethod
    def print_results(results: List[TestResult]):
        print(f"\n\n{'='*70}")
        print("📊 EVALUATION SUMMARY")
        print(f"{'='*70}")
        
        successful_results = [r for r in results if not r.agent_failed]
        total_tests = len(results)
        successful_tests = len(successful_results)
        
        if successful_tests > 0:
            total_assertions = sum(r.total_assertions for r in successful_results)
            passed_assertions = sum(r.passed_assertions for r in successful_results)
            
            print(f"\n  Tests run:      {successful_tests}/{total_tests}")
            print(f"  Assertions:     {passed_assertions}/{total_assertions} passed")
            print(f"  Success Rate:   {(passed_assertions/total_assertions)*100:.1f}%")
            
            failed_assertions = []
            for result in successful_results:
                for i, assertion_result in enumerate(result.assertions_results, 1):
                    if not assertion_result.passed:
                        failed_assertions.append((result.name, assertion_result.assertion))
            
            if failed_assertions:
                print(f"\n{'─'*70}")
                print(f"  ❌ Failed assertions:")
                for name, assertion in failed_assertions:
                    print(f"     • [{name}] {assertion}")
        else:
            print(f"\n  ❌ All {total_tests} tests failed")

## Run the Assertion Evaluator

Below we define two test cases with different assertion counts:
- **Amazon CEO** (4 assertions) — factual, single-answer query
- **Paris Population** (5 assertions) — requires more nuance (city vs metro, source citations, recency)

The evaluator runs the agent, then checks each assertion one-by-one via the LLM judge. Watch the output for which assertions pass and which fail — failures reveal gaps in the agent's response specificity.

In [ ]:
# Function to run the researcher agent
async def researcher_agent_function(query: str) -> AgentResponse:
    try:
        async with researcher_agent.run_mcp_servers():
            result = researcher_agent.run_sync(query)
        return AgentResponse(
            content=result.output.content,
            sources=result.output.sources,
            success=True
        )
    except Exception as e:
        return AgentResponse(
            content="",
            sources=[],
            success=False,
            error=str(e)
        )

# Test cases
test_cases = [
    TestCase(
        name='Amazon CEO Test',
        query='Who is the current CEO of Amazon?',
        assertions=[
            'The response correctly identifies Andy Jassy as the current CEO of Amazon',
            'The response mentions when Andy Jassy became CEO (2021)',
            'The response includes at least one credible source citation',
            'The response is concise and directly answers the question'
        ]
    ),
    TestCase(
        name='Paris Population Test',
        query='What is the population of the capital of France?',
        assertions=[
            'The response correctly identifies Paris as the capital of France',
            'The response provides a specific population figure for Paris',
            'The response distinguishes between city proper and metropolitan area population',
            'The response cites at least two reliable sources',
            'The response indicates when the population data was collected/estimated'
        ]
    )
]

# Usage example
async def run_evaluation():
    print("🚀 Starting Assertion-Based Evaluation\n")
    evaluator = AssertionEvaluator()
    results = await evaluator.evaluate_test_cases(test_cases, researcher_agent_function)
    evaluator.print_results(results)

# Execute
await run_evaluation()

### Interpreting the Results

Look at the failed assertions — they reveal a core eval design challenge:

- **"Distinguishes city proper vs metro"** — the agent gave city-proper population (~2.1M) but the query never asked for metro. Is this a *bug in the agent* or a *bug in the test case*?
- **Non-determinism** — run this cell again and you may get different pass/fail results. LLMs are stochastic; the agent might include the year in one run but not another.

**Rule of thumb:** If an assertion fails >50% of the time across runs, it's testing something the prompt doesn't reliably produce — tighten the prompt or loosen the assertion.

## Externalizing Test Cases

Hard-coding assertions in Python is fine for prototyping, but doesn't scale. In practice you want test cases in a JSON file so they can be:
- Shared across teams (QA, product, engineering)
- Versioned independently of code
- Loaded by different eval frameworks (Ragas, Pydantic Evals, custom harness)

Below: simple export/import utilities using Pydantic's `model_dump()` → JSON.

In [ ]:
from pathlib import Path
from typing import List, Optional
import json

def export_test_cases(test_cases: List[TestCase], file_path: str):
    """Export test cases to JSON file"""
    file_path_obj = Path(file_path)
    file_path_obj.parent.mkdir(parents=True, exist_ok=True)
    
    data = [test_case.model_dump() for test_case in test_cases]
    file_path_obj.write_text(json.dumps(data, indent=2))

def import_test_cases(file_path: str) -> Optional[List[TestCase]]:
    """Import test cases from JSON file"""
    file_path_obj = Path(file_path)
    
    if not file_path_obj.exists():
        return None
    
    try:
        data = json.loads(file_path_obj.read_text())
        return [TestCase(**item) for item in data]
    except:
        return None

# Export test cases
test_cases_path = Path('data/test_cases.json')
export_test_cases(test_cases, str(test_cases_path))

# Import test cases  
loaded_test_cases = import_test_cases(str(test_cases_path))

# View the saved file
print(test_cases_path.read_text())

# Part 2: Step-Based Evaluation

**Concept:** Assertions check *what* the agent said. Step-based eval checks *how* it got there — the sequence of tool calls it made.

**Why this matters:** An agent that makes 8 redundant web searches to answer "Who is the CEO?" is wasting tokens and latency, even if the final answer is correct. Step-based eval catches this.

**The approach:**
1. Run the agent and capture its full message history (tool calls, responses, reasoning)
2. An LLM evaluator groups those raw steps into "logical groupings" (e.g., search + parse = one logical step)
3. Compare logical step count against expected steps → compute **step delta**

**Step delta interpretation:**
- `delta < 0` → agent was *more efficient* than expected
- `delta = 0` → agent matched the gold standard
- `delta > 0` → agent was *less efficient* (possible looping or redundant calls)

First, we re-run the agent while capturing the full message history into `test_runs`.

In [ ]:
from agentic_platform.core.models.memory_models import Message, SessionContext
from pydantic_ai.agent import AgentRunResult
from rich.console import Console
from rich.panel import Panel
from rich.syntax import Syntax
from rich.text import Text
import json
from pydantic_core import to_jsonable_python

console = Console()

test_runs: Dict[str, Any] = {}

# Define the input type for the researcher agent
class ResearchQuery(BaseModel):
    user_query: str

# Updated to use assertion evaluator
async def perform_research_2(query: str) -> AgentResponse:
    try:
        async with researcher_agent.run_mcp_servers():
            result: AgentRunResult[ResearchResponse] = researcher_agent.run_sync(query)
        
        test_runs[query] = result.all_messages()
        
        return AgentResponse(
            content=result.output.content,
            sources=result.output.sources,
            success=True
        )
    except Exception as e:
        return AgentResponse(
            content="",
            sources=[],
            success=False,
            error=str(e)
        )

# Run the assertion evaluation
print("🚀 Starting Assertion-Based Evaluation\n")
evaluator = AssertionEvaluator()
results = await evaluator.evaluate_test_cases(test_cases, perform_research_2)
evaluator.print_results(results)

# Pretty-print the captured message history
def print_message_history(query: str, messages):
    """Format message history with color-coded roles and indented content."""
    console.rule(f"[bold blue]Message History: {query}")
    
    for i, msg in enumerate(messages):
        msg_type = type(msg).__name__
        parts = getattr(msg, 'parts', [])
        
        for part in parts:
            part_type = type(part).__name__
            
            if 'SystemPrompt' in part_type:
                console.print(Panel(
                    part.content[:150] + "..." if len(part.content) > 150 else part.content,
                    title=f"[bold magenta]Step {i+1}: System Prompt",
                    border_style="magenta"
                ))
            
            elif 'UserPrompt' in part_type:
                console.print(Panel(
                    part.content,
                    title=f"[bold green]Step {i+1}: User Query",
                    border_style="green"
                ))
            
            elif 'ToolCall' in part_type:
                args_str = json.dumps(part.args, indent=2) if isinstance(part.args, dict) else str(part.args)
                content = f"[bold]Tool:[/bold] {part.tool_name}\n[bold]Args:[/bold]\n{args_str}"
                console.print(Panel(
                    content,
                    title=f"[bold yellow]Step {i+1}: Tool Call",
                    border_style="yellow"
                ))
            
            elif 'ToolReturn' in part_type:
                raw = part.content
                if isinstance(raw, list):
                    # Tavily search results - show truncated
                    items = []
                    for item in raw[:3]:
                        if isinstance(item, dict):
                            items.append(f"  • {item.get('title', 'N/A')}\n    {item.get('url', '')}\n    Score: {item.get('score', 'N/A'):.3f}")
                    content = "\n".join(items)
                    if len(raw) > 3:
                        content += f"\n  ... +{len(raw)-3} more results"
                else:
                    content = str(raw)[:300]
                
                console.print(Panel(
                    content,
                    title=f"[bold cyan]Step {i+1}: Tool Result ({part.tool_name})",
                    border_style="cyan"
                ))
            
            elif 'Text' in part_type:
                console.print(Panel(
                    part.content,
                    title=f"[bold white]Step {i+1}: Assistant Response",
                    border_style="white"
                ))
            
            elif 'RetryPrompt' in part_type:
                console.print(Panel(
                    part.content,
                    title=f"[bold red]Step {i+1}: Retry (framework forcing structured output)",
                    border_style="red"
                ))
    
    console.print()

console.rule("[bold]Captured Message Histories")
for query, messages in test_runs.items():
    print_message_history(query, messages)

## Convert Messages to Platform Format

The message history from PydanticAI uses its own types. Before we can feed it into our eval harness, we convert it to our platform's canonical `Message` format using `PydanticAIMessageConverter`. This is the same converter used across the platform — it normalizes tool calls, responses, and text into a consistent schema regardless of the underlying agent framework.

In [ ]:
# Now lets convert the message to our types using our converters we've already build. 
from typing import List
from agentic_platform.core.converter.pydanticai_converters import PydanticAIMessageConverter

# Convert the messages to our types.
test_runs_formatted: Dict[str, List[Message]] = {}
for k, v in test_runs.items():
    test_runs_formatted[k] = PydanticAIMessageConverter.convert_messages(v)

# Pretty-print the converted (platform-canonical) messages
def print_converted_messages(query: str, messages: List[Message]):
    """Display converted Message objects with color-coded roles."""
    console.rule(f"[bold blue]Converted Messages: {query}")
    
    for i, msg in enumerate(messages, 1):
        role = msg.role
        
        if role == "system":
            style, label = "magenta", "System"
        elif role == "user":
            style, label = "green", "User"
        elif role == "assistant":
            style, label = "white", "Assistant"
        else:
            style, label = "dim", role.title()
        
        # Build content summary
        parts = []
        if msg.text:
            parts.append(msg.text[:200] + ("..." if len(msg.text) > 200 else ""))
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                args_preview = json.dumps(tc.arguments, indent=2)[:150] if hasattr(tc, 'arguments') else ""
                parts.append(f"[bold]Tool Call:[/bold] {tc.name}\n{args_preview}")
        if hasattr(msg, 'tool_results') and msg.tool_results:
            for tr in msg.tool_results:
                result_preview = str(tr.content)[:150]
                tool_id = tr.id or "unknown"
                parts.append(f"[bold]Tool Result (id={tool_id}):[/bold]\n{result_preview}")
        
        content = "\n".join(parts) if parts else "(empty)"
        
        console.print(Panel(
            content,
            title=f"[bold {style}]Message {i}: {label}",
            border_style=style
        ))
    
    console.print()

console.rule("[bold]Platform-Canonical Message Format")
for k, v in test_runs_formatted.items():
    print_converted_messages(k, v)

## Build the Evaluation Harness

The code below assembles three components:

| Component | Role |
|-----------|------|
| `eval_function` | Sends the agent's message history + expected steps to an LLM evaluator. Uses **forced tool calling** (`force_tool`) to get structured output (`AgentEvalResult`) |
| `run_function` | Runs the agent on a query and returns the converted message history |
| `AgentEvalHarness` | Orchestrates both — runs samples in parallel via `ThreadPoolExecutor`, returns `(sample, result)` pairs |

**Key design choice:** The evaluator LLM is asked to create "logical groupings" of the agent's raw steps, then compare those groupings against the expected steps. This handles non-determinism gracefully — the agent might search differently each run, but the *logical intent* should be stable.

In [ ]:
from labs_common import HAIKU_MODEL_ID
from typing import Literal, Callable
from pydantic import Field
import json
from typing import Tuple
from pydantic_core import to_jsonable_python
from agentic_platform.core.models.llm_models import LLMRequest, LLMResponse
from labs_common import LabsBasePrompt as BasePrompt
from agentic_platform.core.models.tool_models import ToolSpec

from agentic_platform.core.converter.llm_request_converters import ConverseRequestConverter
from agentic_platform.core.converter.llm_response_converters import ConverseResponseConverter
from concurrent.futures import ThreadPoolExecutor
import boto3

SYSTEM_PROMPT = """
You are an expert evaluator of agentic systems. You are given a list of steps the agent took to complete a task and a list of expected steps. 
You need to evaluate the agent's performance based on the expected steps.

You should take the steps inputted and create "logical groupings" of those steps. Those grouping names should come from the expected steps if similar. 
If the agent took a different path, you should create a new grouping name so we can evaluate the agent's performance.

When creating logical groupings, group things together. Instead of saying search the web, gather results. Group them into one step even if the message history shows them as separate steps.
"""

USER_PROMPT = """
Here are the steps the agent took to complete the task:
{steps}

Here are the expected steps:
{expected_steps}

Here is the task success criteria:
{success_criteria}

Before answering, think about the steps and how the agent took them to complete the task.
"""

# Types for the evaluation.

class AgentEvalPrompt(BasePrompt):
    system_prompt: str = SYSTEM_PROMPT
    user_prompt: str = USER_PROMPT

class AgentEvalResult(BaseModel):
    '''Evaluation results for the agent.'''
    thoughts: str = Field(
        description="The evaluators thoughts on the agents performance."
    )
    
    steps: List[str] = Field(
        description="The logical groupings of actions taken by the agent to complete the task, typically representing tool calls or major decision points."
    )
    
    task_success: bool = Field(
        description="Whether the agent successfully completed the task according to the defined success criteria. True indicates success, False indicates failure."
    )

class AgentEvalSample(BaseModel):
    user_input: str
    expected_steps: List[str]
    expected_output: Any
    success_criteria: Literal['Got Corect Answer', '< 1 step from gold standard']


client = boto3.client('bedrock-runtime')

# Helper function to call the LLM.
def call_bedrock(request: LLMRequest) -> LLMResponse:
    kwargs: Dict[str, Any] = ConverseRequestConverter.convert_llm_request(request)
    converse_response: Dict[str, Any] = client.converse(**kwargs)
    return ConverseResponseConverter.to_llm_response(converse_response)


def eval_function(sample: AgentEvalSample, context: List[Message]) -> AgentEvalResult:
    # Create the input for the prompt.
    inputs={
        'steps': json.dumps(to_jsonable_python(context)),
        'expected_steps': '\n'.join(sample.expected_steps),
        'success_criteria': sample.success_criteria
    }

    # Format the prompt.
    prompt: BasePrompt = AgentEvalPrompt(inputs=inputs)

    # Create a tool spec for the structured output.
    tool_spec: ToolSpec = ToolSpec(
        name='AgentEvalResult',
        description='Structured output for an agents performance.',
        model=AgentEvalResult,
    )

    # Create the LLM request and forces structured output through a tool call.
    llm_request: LLMRequest = LLMRequest(
        system_prompt=prompt.system_prompt,
        model_id=HAIKU_MODEL_ID,
        messages=[Message(role='user', text=prompt.user_prompt)],
        hyperparams={"temperature": 0.2},
        tools=[tool_spec],
        force_tool=tool_spec.name
    )

    # Call the LLM and get results.
    llm_response: LLMResponse = call_bedrock(llm_request)
    return AgentEvalResult(**llm_response.tool_calls[0].arguments)

def run_function(sample: AgentEvalSample) -> List[Message]:
    result: AgentRunResult[ResearchResponse] = researcher_agent.run_sync(sample.user_input)
    messages: List[Message] = result.all_messages()
    return PydanticAIMessageConverter.convert_messages(messages)

class AgentEvalHarness:

    def __init__(self, 
                 samples: List[AgentEvalSample], 
                 eval_function: Callable[[AgentEvalSample, List[Message]], AgentEvalResult],
                 run_function: Callable[[AgentEvalSample], List[Message]]):
        
        self.samples = samples
        self.eval_function = eval_function
        self.run_function = run_function

    def evaluate_sample(self, sample: AgentEvalSample) -> Tuple[AgentEvalSample, AgentEvalResult]:
        context: List[Message] = self.run_function(sample)
        result: AgentEvalResult = self.eval_function(sample, context)
        return sample, result
    
    
    def evaluate_threaded(self, num_workers: int = 2) -> List[AgentEvalSample]:
        with ThreadPoolExecutor(max_workers=num_workers) as executor:
            results: List[Tuple[AgentEvalSample, AgentEvalResult]] = list(executor.map(self.evaluate_sample, self.samples))
        return results





In [ ]:
samples = [
    AgentEvalSample(
        user_input='Who is the current CEO of Amazon?',
        expected_steps=['Search the web for the current CEO of Amazon', "Verify the information is correct"],
        expected_output='Andy Jassy',
        success_criteria='Got Corect Answer'
    ),
    AgentEvalSample(
        user_input='What is the population of the capital of France?',
        expected_steps=[
            'Search the web for the capital of France',
            'Search the web for the population of Paris',
            'Format results into a response'
        ],
        expected_output='Roughly 2.1 million',
        success_criteria='Got Corect Answer'
    )
]

harness = AgentEvalHarness(samples, eval_function, run_function)
evaluation_results = harness.evaluate_threaded()
evaluation_results

## Summarizing Results

Reading raw evaluator output per-sample doesn't scale. Below we create a summary function that computes two key metrics across all samples:
- **Success rate** — percentage of tasks where `task_success=True`
- **Average step delta** — how many more (or fewer) steps the agent took vs expected, averaged across all samples

In [ ]:
def create_evaluation_summary(eval_pairs: List[Tuple[AgentEvalSample, AgentEvalResult]]) -> str:
    total_samples = len(eval_pairs)
    success_count = 0
    total_step_delta = 0
    
    for sample, result in eval_pairs:
        # Calculate step delta
        expected_step_count = len(sample.expected_steps)
        actual_step_count = len(result.steps)
        step_delta = actual_step_count - expected_step_count
        
        # Update totals
        if result.task_success:
            success_count += 1
        total_step_delta += step_delta
    
    # Calculate averages
    success_rate = success_count / total_samples * 100
    avg_step_delta = total_step_delta / total_samples
    
    # Generate summary
    summary = f"""
    AGENT EVALUATION SUMMARY
    ========================
    Total samples evaluated: {total_samples}
    Success rate: {success_rate:.1f}%
    Average step delta: {avg_step_delta:.2f}

    Step delta interpretation:
    - Negative: Agent used fewer steps than expected (more efficient)
    - Zero: Agent used exactly the expected number of steps
    - Positive: Agent used more steps than expected (less efficient)
    """
    
    return summary

In [ ]:
print(create_evaluation_summary(eval_pairs=evaluation_results))

### Reading the Step Delta

The summary above shows **avg step delta = 0.50** — meaning on average, the agent used half a step more than expected. Use this rules-of-thumb table to interpret step delta values across your eval runs:

| Avg Step Delta | Interpretation | Action |
|---------------|----------------|--------|
| **-1 to 0** | Agent is *efficient* — combining steps or skipping unnecessary ones | None needed; validate the shortcuts are safe |
| **0 to +1** | Normal variance — evaluator grouping may differ slightly from expectations | Acceptable; monitor for trends |
| **+2 or more** | Investigate — agent may be looping or making redundant calls | Review traces; tighten the prompt or add guardrails |
| **+5 or more** | Red flag — likely a bug (retry loops, repeated searches, stuck in a cycle) | Fix immediately; this is burning tokens and latency |

**In CI/CD:** Set a threshold (e.g., fail the build if avg step delta > 2). This catches regressions where a prompt change causes the agent to loop without affecting correctness.

## DataFrame View: Per-Query Drill-Down

The summary gives you top-line metrics; the DataFrame lets you drill into individual samples. This is where you spot patterns: which queries consistently have high step delta? Which have low success? In a CI pipeline, you'd assert thresholds here (e.g., fail the build if avg step delta > 2 or success rate < 90%).

In [ ]:
import pandas as pd
from typing import List, Tuple

def create_evaluation_dataframe(eval_pairs: List[Tuple]):
    """Convert evaluation pairs to a structured pandas DataFrame."""
    
    data = []
    for sample, result in eval_pairs:
        # Calculate step delta
        expected_step_count = len(sample.expected_steps)
        actual_step_count = len(result.steps)
        step_delta = actual_step_count - expected_step_count
        
        # Create row
        row = {
            'user_input': sample.user_input,
            'expected_steps': sample.expected_steps,
            'expected_output': sample.expected_output,
            'success_criteria': sample.success_criteria,
            'actual_steps': result.steps,
            'step_count': actual_step_count,
            'expected_step_count': expected_step_count,
            'step_delta': step_delta,
            'task_success': result.task_success,
            'thoughts': result.thoughts
        }
        data.append(row)
    
    # Create DataFrame
    df = pd.DataFrame(data)
    
    # Calculate summary statistics and add to dataframe attributes
    df.attrs['total_samples'] = len(eval_pairs)
    df.attrs['success_rate'] = df['task_success'].mean() * 100
    df.attrs['avg_step_delta'] = df['step_delta'].mean()
    
    return df

# Example usage:
df = create_evaluation_dataframe(evaluation_results)

# Display basic info
print(f"Evaluation results for {df.attrs['total_samples']} samples:")
print(f"Success rate: {df.attrs['success_rate']:.1f}%")
print(f"Average step delta: {df.attrs['avg_step_delta']:.2f}")

# Display the DataFrame
print("\nDetailed results:")
print(df[['user_input', 'step_delta', 'task_success']])

# Output summary statistics
print("\nSummary by success status:")
print(df.groupby('task_success')['step_delta'].describe())

# From Scratch to Production: Eval Frameworks

We built ~200 lines of plumbing to evaluate a single agent. In production, **eval frameworks reduce this to test case definitions + config** — they handle the harness, parallelism, retries, dashboards, and CI/CD integration for you.

## Framework Landscape

| Category | Framework | What it handles |
|----------|-----------|----------------|
| **AWS-Native** | Amazon Bedrock Evaluations | Managed eval jobs (accuracy, toxicity, custom metrics) — no infra to manage |
| **AWS-Native** | Amazon Bedrock AgentCore Evaluations | Agent-specific evals (tool selection, multi-turn, step efficiency) |
| **Open-Source** | Promptfoo | Config-driven eval (YAML test cases), CI/CD-native, supports any LLM provider |
| **Open-Source** | Ragas | RAG-focused (faithfulness, relevance, context recall) — great for search agents |
| **Open-Source** | Pydantic Evals | Python-native, type-safe test cases, integrates with Pydantic models |
| **Open-Source** | DeepEval | Pytest-style LLM evals with 14+ metrics out of the box |

## CI/CD Integration Pattern

```
prompt change → trigger eval suite → compare metrics vs baseline
    → regression detected? → block merge + alert
    → metrics stable? → auto-approve + deploy
```

**What frameworks give you that hand-built code doesn't:**
- Automatic retries and statistical significance (run N times, report confidence intervals)
- Dashboard + historical tracking (detect slow regressions over weeks)
- A/B comparison (evaluate prompt A vs prompt B on the same test set)
- Team collaboration (QA writes test cases in YAML/JSON, engineers own the harness config)

## Why We Built It From Scratch First

Understanding the internals — LLM-as-judge, forced structured output, logical groupings, step delta — means you can:
1. **Debug** when a framework gives unexpected results
2. **Extend** with custom evaluators when built-in metrics don't fit
3. **Interpret** results with confidence instead of treating the framework as a black box

**Bottom line:** Use a framework for production. Use the patterns in this notebook to understand what the framework is doing under the hood.

# Conclusion

**What we built:**
- An assertion-based evaluator (LLM-as-judge, TRUE/FALSE per assertion)
- A step-based evaluator (logical groupings, step delta metric)
- Export/import for test cases (JSON, framework-agnostic)
- Summary + DataFrame utilities for tracking metrics

**Key takeaways:**
1. **Assertions test correctness, steps test efficiency** — use both together
2. **Non-determinism is expected** — agents may pass 8/9 assertions one run and 9/9 the next. Track trends, not individual runs
3. **Step delta is a regression signal** — if it trends upward after a prompt or model change, investigate before shipping
4. **In production, use a framework** — the plumbing code above exists in every eval framework. Pick one (Promptfoo, Ragas, Bedrock Evaluations) and focus on writing good test cases, not infrastructure